In [ ]:
dbutils.widgets.text("catalog", "adb_adventureworks_project")
dbutils.widgets.text("silver_schema", "silver_dev")
dbutils.widgets.text("gold_schema", "gold_dev")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

silver = f"{catalog}.{silver_schema}"
gold = f"{catalog}.{gold_schema}"

In [0]:
fact_table = f"{gold}.fact_sales"

df = spark.table(fact_table)

total_rows = df.count()

unique_lines = (
    df.select("OrderNumber", "OrderLineItem")
      .distinct()
      .count()
)

assert total_rows == unique_lines, (
    f"Duplicate sales lines detected! "
    f"Total: {total_rows}, Unique: {unique_lines}"
)

print(f"PASS: {total_rows} unique sales lines")


## Revenue Reconciliation

In [0]:
from pyspark.sql import functions as F

# Calculate expected revenue from current-state Silver
silver_sales = spark.table(f"{silver}.sales_current_view")
silver_products = spark.table(f"{silver}.products_silver_table")

expected_revenue = (
    silver_sales.alias("s")
    .join(
        silver_products.alias("p"),
        F.col("s.ProductKey") == F.col("p.ProductKey"),
        "inner"
    )
    .select(
        (
          F.col("s.OrderQuantity") * F.col("p.ProductPrice")
        ).cast("decimal(18,4)").alias("Revenue")
    )
    .agg(F.sum("Revenue").alias("TotalRevenue"))
    .first()["TotalRevenue"]
)

# Calculate actual revenue from Gold
actual_revenue = (
    spark.table(f"{gold}.fact_sales")
    .agg(F.sum("EstimatedRevenue").alias("TotalRevenue"))
    .first()["TotalRevenue"]
)

# assert condition, error_message
assert round(expected_revenue, 2) == round(actual_revenue, 2), (
    f"Revenue mismatch! "
    f"Silver: {expected_revenue}, "
    f"Gold: {actual_revenue}"
)

print("PASS: Silver and Gold revenue reconciled")
print(f"Revenue: {actual_revenue:.2f}")


In [0]:
from pyspark.sql import functions as F

fact = spark.table(f"{gold}.fact_sales")
monthly = spark.table(f"{gold}.agg_monthly_sales")

# Calculate total revenue from fact table
fact_revenue = (
    fact.agg(
        F.sum("EstimatedRevenue").alias("revenue")
    )
    .first()["revenue"]
)

# Calculate total revenue from monthly aggregate
monthly_revenue = (
    monthly.agg(
        F.sum("EstimatedRevenue").alias("revenue")
    )
    .first()["revenue"]
)

# Validate reconciliation
assert fact_revenue == monthly_revenue, (
    f"Revenue mismatch: "
    f"fact_sales={fact_revenue}, "
    f"agg_monthly_sales={monthly_revenue}"
)

print("PASS: Gold fact and monthly aggregate revenue reconciled")
print(f"Estimated revenue: {fact_revenue:.2f}")